In [ ]:
import json

import pandas as pd
from IPython.display import display

from study.data import observation_batches
from study.features import build_features
from study.io import RESOURCES, WORK, save, write_table

In [ ]:
TIERS = ('R0', 'R1', 'R2', 'R3', 'R4')

In [ ]:
# Build feature matrices for each cohort
for name in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    parts = [build_features(batch) for batch in observation_batches(name)]
    matrix = pd.concat(parts).sort_index()
    
    write_table(WORK / 'features' / f'{name}.parquet', matrix)
    
    print(name, matrix.shape)

In [ ]:
# Build the nested R0-R4 tiers
dictionary = pd.read_csv(RESOURCES / 'feature_dictionary.csv')
primary = dictionary.loc[dictionary.primary_axis_feature]
static = set(primary.loc[primary.axis.eq('static'), 'feature'])

# Define static and last-value tiers
r0 = static | {f'{name}_last' for name in ('HR', 'RespRate', 'Temp', 'SaO2', 'MAP', 'NIMAP')}
last = {
    name for name in dictionary.feature if name.endswith('_last') and (not name.startswith(('MechVent_', 'Urine_')))
}
r1 = r0 | last | {'Urine_total', 'MechVent_ever'}

# Add physiological and observation summaries
r2 = r1 | set(primary.loc[primary.axis.isin(['severity', 'trajectory', 'variability']), 'feature'])
r3 = r2 | set(primary.loc[primary.axis.eq('observation_process'), 'feature'])
r4 = set(dictionary.feature)

representations = {
    name: [feature for feature in dictionary.feature if feature in members]
    for name, members in zip(TIERS, (r0, r1, r2, r3, r4), strict = True)
}

In [ ]:
# Load shared feature sets for external comparisons
shared = json.loads((RESOURCES / 'mappings/shared_representations.json').read_text())
shared = {name.removesuffix('_shared'): columns for name, columns in shared.items()}

# Save feature sets and summarise tier sizes
save(WORK / 'feature_sets.pkl', {'original': representations, 'shared': shared})

display(
    pd.DataFrame(
        {
            support: {tier: len(columns) for tier, columns in tiers.items()}
            for support, tiers in [('original', representations), ('shared', shared)]
        }
    )
)